# Contact Features and Empty-Car Hunting

I test whether wheelset mismatch and contact measurements add predictive
information beyond mean conicity and rail profile. I compare the same kernel
ridge model across four feature sets and hold out whole cars during evaluation.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import pandas as pd

project_folder = ROOT

from contact_hunting import (
    load_tables, prepare_trucks, evaluate_models, paired_car_intervals
)

data_folder = DATA


## Data

The inputs are the team's merged table and the contact-point difference CSV.
I combine the two wheelsets in each truck into one truck/rail assessment.
The loading function checks the matches, axle identifiers, and repeated truck results.


In [ ]:
merged_data, contact_data = load_tables(data_folder)
truck_data = prepare_trucks(merged_data, contact_data)

print("Truck/rail assessments:", len(truck_data))
print("Cars:", truck_data["Car"].nunique())
print("Trucks:", len(truck_data[["Car", "Truck"]].drop_duplicates()))


## Hunting criteria

Acceleration is the existing truck-level target. I also evaluate displacement
as a separate sensitivity comparison. The displacement label uses the lower of
the two axle speeds, which matches the supplied truck-speed summary. A speed
below 80 mph counts as hunting; 80 means no hunting was recorded within the
tested range.

In the table, 0 means no hunting recorded and 1 means hunting.


In [ ]:
target_agreement = pd.crosstab(
    truck_data["acceleration_hunted"],
    truck_data["displacement_hunted"],
    rownames=["Acceleration criterion"],
    colnames=["Displacement criterion"],
)
display(target_agreement)

disagreement_count = (
    truck_data["acceleration_hunted"] != truck_data["displacement_hunted"]
).sum()
print("Assessments where the criteria disagree:", disagreement_count)


## Feature comparisons

I add predictors in stages:

| Model | Predictors |
|---|---|
| Rail kernel | Rail profile |
| Mean conicity + rail | Rail profile and mean wheelset conicity |
| Conicity + rail | Add the absolute difference between wheelset conicities |
| Conicity + contact + rail | Add the three contact measurements |

The contact measurements are profile difference, lateral gradient, and
contact-angle difference. I take the larger value across the wheelsets and
apply `log1p` to each. Raw measurements stay unchanged.

The four models use the same Gaussian-distance scale and parameter grid. Each
outer fold holds out one car. Three grouped folds within the remaining cars
select the model settings; numeric scaling uses training rows only. I also
include fixed always-hunting and per-rail-majority baselines.


In [ ]:
target_names = {
    "acceleration_hunted": "Acceleration",
    "displacement_hunted": "Displacement sensitivity",
}

metric_tables = []
interval_tables = []
comparison_tables = []
local_predictions = {}

for target, label in target_names.items():
    metrics, predictions, settings = evaluate_models(truck_data, target)
    intervals, comparisons = paired_car_intervals(
        predictions, n_bootstrap=2000, seed=42
    )
    metrics["criterion"] = label
    intervals["criterion"] = label
    comparisons["criterion"] = label
    metric_tables.append(metrics)
    interval_tables.append(intervals)
    comparison_tables.append(comparisons)
    local_predictions[target] = predictions

results = pd.concat(metric_tables, ignore_index=True)
score_intervals = pd.concat(interval_tables, ignore_index=True)
comparisons = pd.concat(comparison_tables, ignore_index=True)


## Performance on held-out cars

Balanced accuracy averages recall for hunting and no hunting. The ranges come
from 2,000 resamples of whole cars, using the same draws for each model. The
predictions stay fixed, so the ranges exclude uncertainty from retraining and
exploratory feature selection.

In [ ]:
performance = results.merge(
    score_intervals[["criterion", "model", "lower_95", "upper_95"]],
    on=["criterion", "model"],
    validate="one_to_one",
).set_index(["criterion", "model"])

performance_table = (100 * performance[
    ["balanced_accuracy", "recall", "specificity"]
]).round(1).rename(columns={
    "balanced_accuracy": "Balanced accuracy (%)",
    "recall": "Hunting recall (%)",
    "specificity": "No-hunting recall (%)",
})
performance_table["95% car-bootstrap range (%)"] = [
    f"{100 * lower:.1f} to {100 * upper:.1f}"
    for lower, upper in zip(performance.lower_95, performance.upper_95)
]
display(performance_table)


## Change in balanced accuracy

The table gives paired changes in percentage points. A range that crosses zero
leaves the direction of the change uncertain. The last row compares the full
model directly with mean conicity and rail profile.

In [ ]:
comparison_table = comparisons.set_index(["criterion", "comparison"])
comparison_display = (100 * comparison_table[
    ["change_in_balanced_accuracy", "lower_95", "upper_95"]
]).round(1).rename(columns={
    "change_in_balanced_accuracy": "Change (percentage points)",
    "lower_95": "95% lower",
    "upper_95": "95% upper",
})
display(comparison_display)


## Results

Mean conicity improved on the rail-only kernel. Adding the contact measurements
and wheelset conicity difference to that baseline barely improved the
acceleration score and lowered the displacement score. Both ranges crossed
zero. The contact measurements helped when added to the weaker mismatch model,
but that comparison starts from a different baseline.